## Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

## Load Libraries

In [3]:
library(openxlsx)
library(readxl)
library(dplyr)
library(glue)
library(ggplot2)
library(viridis)
library(ggrepel)

# Read in Data

In [6]:
# Initialize an empty dataframe
all_data <- data.frame()

# Specify the directory containing the files
directory <- "06_DEG_GO"

# List all files in the directory that match the pattern
file_list <- list.files(path = directory, pattern = "results_enrichr_results\\.xlsx$", full.names = TRUE)

# Iterate over each file and read in the corresponding Excel sheets
for (file_path in file_list) {
  # Extract the comparison name from the file name
  comparison <- tools::file_path_sans_ext(basename(file_path))
      
  # Get the names of the sheets in the Excel file
  databases <- excel_sheets(file_path)
  
  try({
    # Read the data from the Excel file
    df <- read_excel(file_path, sheet = "RNAseq_DiseaseGene_DrugSigs_GEO")
    
    if (nrow(df) == 0) {
      next
    }
    
    # Select the required columns and add module (comparison) and database information
    df <- df %>%
      select(Term, Adjusted.P.value, Odds.Ratio) %>%
      mutate(Module = comparison, Database = "RNAseq_DiseaseGene_DrugSigs_GEO")

    # Filter out rows where the Adjusted.P.value is less than or equal to 0.05
    df <- df %>%
    filter(Adjusted.P.value <= 0.05)
      
    # Append to the dataframe
    all_data <- bind_rows(all_data, df)
    
  }, silent = TRUE)
  }

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Rename
all_data$Module <- gsub("_results_enrichr_results$", "", all_data$Module)

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Bardet-Biedi Syndrome Neurons (IPSCs-derived) GSE62936 up,0.003874651,2.711731,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
2,MeCP2 Hypothalamus Knockout GSE66871 up,0.006233866,2.590074,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
3,Gsa Cerebellum knockout GSE60415 down,0.007351239,2.259899,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
4,LPS Neuron GSE75246 down,0.007351239,2.509386,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
5,Cocaine Nucleus accumbens 20 mg- kg GSE42805 up,0.011511441,2.428376,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
6,HDAC3 Hippocampus Knockout GSE72196 up,0.011511441,2.401911,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO


In [7]:
# Count terms for human data
term_counts <- all_data %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term),
    .groups = "drop"
  )

# Print results
print(term_counts)

# A tibble: 1 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 RNAseq_DiseaseGene_DrugSigs_GEO         192          136


In [8]:
# Plot terms with key words
keywords <- c("alzheimer", "hd", "huntington", "neurodegenerat", "parkinson", "mecp2")

# Create a pattern for grepl
pattern <- paste(keywords, collapse = "|")

# Filter all_data
all_data <- all_data[grepl(pattern, all_data$Term, ignore.case = TRUE), ]

# View remaining terms for human data
term_counts <- all_data %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term),
    .groups = "drop"
  )

# Print results
print(term_counts)
head(all_data)

# A tibble: 1 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 RNAseq_DiseaseGene_DrugSigs_GEO          18           13


,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
2,MeCP2 Hypothalamus Knockout GSE66871 up,6.233866e-03,2.590074,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
6,HDAC3 Hippocampus Knockout GSE72196 up,1.151144e-02,2.401911,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
14,HD (R6/1; transgenic for exon 1) Striatum Mutant GSE59571 down,2.670653e-02,2.216104,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
17,MeCP2 Hypothalamus Transgenic GSE66871 down,2.670653e-02,2.206501,MUT_Guide6_vs_MUT_NonEffector,RNAseq_DiseaseGene_DrugSigs_GEO
20,HD (R6/1; transgenic for exon 1) Striatum Mutant GSE59571 down,1.264215e-11,4.279545,MUT_Guide6_vs_MUT_NT,RNAseq_DiseaseGene_DrugSigs_GEO
26,MeCP2 Hypothalamus Knockout GSE66871 down,1.534208e-07,3.404480,MUT_Guide6_vs_MUT_NT,RNAseq_DiseaseGene_DrugSigs_GEO


# RNA-seq Disease Gene and Drug Signatures from GEO Dot Plot

In [17]:
# Plot
dot_plot <- ggplot(all_data, aes(x = gsub("_", " ", Module), y = Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
  geom_point(shape = 21) +
  scale_fill_viridis() +
  xlab('') + ylab('') +
  labs(title = "RNA-seq Disease Gene and\nDrug Signatures from GEO") +
  theme(
    panel.background = element_rect(fill = "white", color = NA),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    plot.background = element_rect(fill = "white", color = NA),
    axis.text.x = element_text(angle = 90, hjust = 1),
    axis.text.y = element_text(size = 8),
    axis.title = element_text(size = 8)
  )

# Save
ggsave(filename = "00_FIGURES/filtered_RNAseq_DiseaseGene_DrugSigs_GEO.pdf", plot = dot_plot, height = 6, width = 8, limitsize = FALSE)